## 1. Imports and plotting defaults


## Minimal software dependencies
 at least:

```text
anndata
scanpy
pandas
numpy
scipy
scikit-learn
matplotlib
seaborn
igraph
leidenalg
```


In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
from sklearn.decomposition import non_negative_factorization

RANDOM_STATE = 42
sc.settings.verbosity = 2

# Keep text editable in exported vector PDFs.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["savefig.transparent"] = True


## 2. Editable paths and analysis parameters


### Repository assumptions

This notebook is intended to run from a pre-filtered AnnData `.h5ad` object whose `.X` matrix contains the myeloid cells to analyze. If starting from a whole tumor/all-cell object, subset to the desired myeloid cells before running this notebook.

Required files in the working directory or supplied as paths below:
- input `.h5ad` object
- `Myeloid_NMF_Average_Gene_Spectra.txt` reference spectra file from the Miller et al. back-calculation workflow


In [ ]:
ADATA_PATH = Path("path_to_your_pre_filtered_myeloid_object.h5ad")
SPECTRA_PATH = Path("./Myeloid_NMF_Average_Gene_Spectra.txt")
# This Average Gene Spectra file is available from the BernsteinLab/Calculate_Myeloid_cNMF_Usage github repository: https://github.com/BernsteinLab/Calculate_Myeloid_cNMF_Usage under Required Files

UMAP_OUT_DIR = Path("program_usage_umaps_pdf")
PRE_PANEL_OUT_DIR = Path("pre_only_program_usage_panels_pdf")
UMAP_OUT_DIR.mkdir(parents=True, exist_ok=True)
PRE_PANEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_OBS_COLUMNS = ("sample_name", "response_status")

N_COMPONENTS = 14
N_TOP_HVG = 4000
LEIDEN_RESOLUTION = 0.7
UMAP_USAGE_VMAX = 50

SHOW_PLOTS = True
SAVE_PDF = True


## 3. Helper functions


In [ ]:

def validate_inputs(
    adata_path: Path,
    spectra_path: Path,
    required_obs_columns: tuple[str, ...] = ("sample_name", "response_status"),
) -> None:
    """Fail early if required repository inputs are missing."""
    missing_files = [str(p) for p in (adata_path, spectra_path) if not p.exists()]
    if missing_files:
        raise FileNotFoundError(
            "Missing required input file(s): " + ", ".join(missing_files)
        )


def validate_adata_schema(adata: ad.AnnData, required_obs_columns: tuple[str, ...]) -> None:
    """Validate expected AnnData fields before analysis/plotting."""
    missing = [col for col in required_obs_columns if col not in adata.obs.columns]
    if missing:
        raise KeyError(
            "Missing required adata.obs column(s): " + ", ".join(missing)
        )
    if adata.n_obs == 0 or adata.n_vars == 0:
        raise ValueError("Input AnnData has zero cells or zero genes.")


def ensure_scanpy_obsm_keys(adata: ad.AnnData) -> ad.AnnData:
    """
    Convert zellkonverter-style dimensional-reduction keys to Scanpy conventions
    when needed.
    """
    if "UMAP" in adata.obsm and "X_umap" not in adata.obsm:
        adata.obsm["X_umap"] = np.asarray(adata.obsm["UMAP"])
    if "PCA" in adata.obsm and "X_pca" not in adata.obsm:
        adata.obsm["X_pca"] = np.asarray(adata.obsm["PCA"])
    return adata


def normalize_data_before_backcalculation(adata: ad.AnnData) -> ad.AnnData:
    """
    Normalize the cell-by-gene matrix as expected for cNMF program back-calculation:
    scale each gene to unit variance without zero-centering sparse matrices.
    """
    adata_normalized = adata.copy()

    if sp.issparse(adata_normalized.X):
        sc.pp.scale(adata_normalized, zero_center=False)
        if np.isnan(adata_normalized.X.data).sum() > 0:
            print("[warning] NaNs detected in normalized sparse matrix.")
    else:
        gene_sd = adata_normalized.X.std(axis=0, ddof=1)
        gene_sd[gene_sd == 0] = 1
        adata_normalized.X = adata_normalized.X / gene_sd
        if np.isnan(adata_normalized.X).sum() > 0:
            print("[warning] NaNs detected in normalized dense matrix.")

    return adata_normalized


def backcalculate_myeloid_program_usage(
    adata_normalized: ad.AnnData,
    myeloid_program_matrix: pd.DataFrame,
    n_components: int = 14,
    random_state: int | None = None,
) -> pd.DataFrame:
    """
    Back-calculate reference myeloid program usage in a new AnnData object.
    """
    adata_normalized.obs_names_make_unique()

    X_df = pd.DataFrame(
        data=adata_normalized.X.toarray() if sp.issparse(adata_normalized.X) else adata_normalized.X,
        columns=adata_normalized.var_names,
        index=adata_normalized.obs_names,
    )

    spectra_t = myeloid_program_matrix.T
    matched_spectra_t = spectra_t.filter(items=X_df.columns)
    matched_X = X_df.filter(items=matched_spectra_t.columns)

    if matched_X.shape[1] == 0:
        raise ValueError("No overlapping genes were found between `adata` and `myeloid_program_matrix`.")

    W, H, n_iter = non_negative_factorization(
        matched_X.to_numpy(dtype=np.float64),
        W=None,
        H=matched_spectra_t.to_numpy(dtype=np.float64),
        n_components=n_components,
        init="random",
        update_H=False,
        solver="cd",
        beta_loss="frobenius",
        tol=1e-4,
        max_iter=1000,
        alpha_W=0.0,
        alpha_H="same",
        l1_ratio=0.0,
        random_state=random_state,
        verbose=0,
        shuffle=False,
    )

    usage = pd.DataFrame(W, columns=myeloid_program_matrix.columns, index=adata_normalized.obs_names)
    row_sums = usage.sum(axis=1).replace(0, np.nan)
    usage_pct = usage.div(row_sums, axis=0).mul(100).fillna(0)

    print(f"[done] Back-calculated usage for {usage_pct.shape[0]:,} cells and {usage_pct.shape[1]} programs.")
    return usage_pct


def build_sample_order(adata: ad.AnnData, sample_col: str = "sample_name") -> list[str]:
    """
    Order samples by patient number and then Pre before Post, falling back to
    lexicographic order if patient/timepoint parsing is unavailable.
    """
    samples = pd.Index(adata.obs[sample_col].dropna().astype(str).unique())

    patient_num = samples.str.extract(r"^P(\d+)_", expand=False)
    timepoint = samples.str.extract(r"_(Pre|Post)$", expand=False)

    if patient_num.notna().any():
        patient_key = pd.to_numeric(patient_num, errors="coerce").fillna(np.inf).to_numpy()
        timepoint_key = timepoint.map({"Pre": 0, "Post": 1}).fillna(99).astype(int).to_numpy()
        order = np.lexsort((timepoint_key, patient_key))
        return samples[order].tolist()

    return sorted(samples.tolist())


def safe_filename(value: str) -> str:
    """Return a filesystem-safe version of a program/sample name."""
    return "".join(c if c.isalnum() or c in "._-+" else "_" for c in str(value))


## 4. Load input data


In [ ]:
validate_inputs(ADATA_PATH, SPECTRA_PATH, REQUIRED_OBS_COLUMNS)

adata = sc.read_h5ad(ADATA_PATH)
adata = ensure_scanpy_obsm_keys(adata)
validate_adata_schema(adata, REQUIRED_OBS_COLUMNS)

# This notebook assumes ADATA_PATH already points to the myeloid-cell object to analyze.
adata_myeloid = adata
adata_myeloid_adjusted = adata.copy()

original_glioma_spectra = pd.read_table(SPECTRA_PATH, sep="\t", index_col=0)

print(adata_myeloid_adjusted)
print(original_glioma_spectra.shape)


## 5. Back-calculate myeloid program usage


In [ ]:
adata_myeloid_normalized = normalize_data_before_backcalculation(adata_myeloid_adjusted)

adata_myeloid_bc_usages = backcalculate_myeloid_program_usage(
    adata_myeloid_normalized,
    original_glioma_spectra,
    n_components=N_COMPONENTS,
    random_state=RANDOM_STATE,
)

adata_myeloid_bc_usages = adata_myeloid_bc_usages.loc[adata_myeloid_adjusted.obs_names]

adata_myeloid_adjusted.obsm["program_usage"] = adata_myeloid_bc_usages.values
adata_myeloid_adjusted.uns["program_names"] = list(adata_myeloid_bc_usages.columns)

prog_names = list(adata_myeloid_bc_usages.columns)
usage = adata_myeloid_bc_usages
progs = prog_names

adata_myeloid_bc_usages.head()


## 6. cluster myeloid cells and compute UMAP


In [ ]:
adata_myeloid_adjusted_cluster = adata_myeloid_adjusted.copy()
adata_myeloid_adjusted_cluster.layers["counts"] = adata_myeloid_adjusted_cluster.X.copy()

sc.pp.normalize_total(adata_myeloid_adjusted_cluster)
sc.pp.log1p(adata_myeloid_adjusted_cluster)

sc.pp.highly_variable_genes(
    adata_myeloid_adjusted_cluster,
    n_top_genes=N_TOP_HVG,
    batch_key="sample_name",
)

sc.tl.pca(adata_myeloid_adjusted_cluster, random_state=RANDOM_STATE)
sc.pp.neighbors(adata_myeloid_adjusted_cluster, random_state=RANDOM_STATE)
sc.tl.umap(adata_myeloid_adjusted_cluster, random_state=RANDOM_STATE)
sc.tl.leiden(
    adata_myeloid_adjusted_cluster,
    flavor="igraph",
    n_iterations=2,
    resolution=LEIDEN_RESOLUTION,
    random_state=RANDOM_STATE,
)


## 7. General clustering/metadata UMAPs


In [ ]:
sc.pl.umap(
    adata_myeloid_adjusted_cluster,
    color=["leiden"],
    legend_loc="on data",
    legend_fontoutline=2,
    show=SHOW_PLOTS,
)

sc.pl.umap(
    adata_myeloid_adjusted_cluster,
    color=["sample_name"],
    legend_fontoutline=2,
    show=SHOW_PLOTS,
)

for obs_col in ["cell_type_manual", "timepoint", "response_status"]:
    if obs_col in adata_myeloid_adjusted_cluster.obs.columns:
        sc.pl.umap(
            adata_myeloid_adjusted_cluster,
            color=[obs_col],
            legend_fontoutline=2,
            show=SHOW_PLOTS,
        )


## 8. UMAPs colored by back-calculated program usage


In [ ]:
for prog in prog_names:
    col = f"{prog}_usage"

    adata_myeloid_adjusted_cluster.obs[col] = (
        adata_myeloid_adjusted_cluster.obs_names
        .to_series()
        .map(adata_myeloid_bc_usages[prog])
        .fillna(0)
        .astype(float)
    )

    sc.pl.umap(
        adata_myeloid_adjusted_cluster,
        color=col,
        cmap="viridis",
        title=f"Usage of {prog}",
        vmax=UMAP_USAGE_VMAX,
        show=SHOW_PLOTS,
    )

    if SAVE_PDF:
        ax = sc.pl.umap(
            adata_myeloid_adjusted_cluster,
            color=col,
            cmap="viridis",
            title=f"Usage of {prog}",
            vmax=UMAP_USAGE_VMAX,
            show=False,
            return_fig=False,
        )
        fig = ax.figure
        out_path = UMAP_OUT_DIR / f"{safe_filename(prog)}__usage_umap.pdf"
        fig.savefig(out_path, format="pdf", bbox_inches="tight")
        plt.close(fig)
        print(f"[saved] {out_path}")


## 9. program usage panels: violin, box, sample mean dots


In [ ]:
palette = {"responder": "#1f77b4", "nonresponder": "#d62728"}
group_order = ["responder", "nonresponder"]
sample_order = build_sample_order(adata, sample_col="sample_name")


def _set_violin_alpha(ax, alpha: float = 0.35) -> None:
    """
    Seaborn violinplot does not reliably honor alpha across versions.
    This sets alpha on violin PolyCollections only.
    """
    import matplotlib.collections as mcoll

    for coll in ax.collections:
        if isinstance(coll, mcoll.PolyCollection):
            coll.set_alpha(alpha)


def build_pre_mask_and_order(
    adata: ad.AnnData,
    sample_order: list[str],
    sample_col: str = "sample_name",
    timepoint_col: str = "timepoint",
    patient_col: str = "patient",
) -> tuple[pd.Series, list[str]]:
    """Create a Pre-only mask and ordered Pre sample list."""
    if timepoint_col in adata.obs.columns:
        pre_mask = adata.obs[timepoint_col].astype(str).str.lower().eq("pre")
    else:
        pre_mask = adata.obs[sample_col].astype(str).str.contains(r"_Pre\b", regex=True)

    pre_samples = adata.obs.loc[pre_mask, sample_col].dropna().unique().tolist()
    sample_order_pre = [s for s in sample_order if s in set(pre_samples)]

    if len(sample_order_pre) == 0:
        order_cols = [sample_col]
        if patient_col in adata.obs.columns:
            order_cols.append(patient_col)

        tmp = adata.obs.loc[pre_mask, order_cols].dropna().drop_duplicates().copy()
        if patient_col in tmp.columns:
            tmp["patient_num"] = tmp[patient_col].astype(str).str.extract(r"(\d+)", expand=False).astype(float)
            tmp = tmp.sort_values(["patient_num", patient_col, sample_col])
        else:
            tmp = tmp.sort_values(sample_col)

        sample_order_pre = tmp[sample_col].tolist()

    return pre_mask, sample_order_pre


def plot_pre_program_usage_panels(
    adata: ad.AnnData,
    usage: pd.DataFrame,
    progs: list[str],
    sample_order: list[str],
    out_dir: Path,
    palette: dict[str, str],
    group_order: list[str],
    show_plots: bool = True,
    save_pdf: bool = True,
) -> None:
    """Export one editable vector PDF per program for Pre-only per-cell usage."""
    pre_mask, sample_order_pre = build_pre_mask_and_order(adata, sample_order)

    for prog in progs:
        dfp = adata.obs.loc[pre_mask, ["sample_name", "response_status"]].copy()

        if isinstance(usage, pd.DataFrame):
            u = pd.to_numeric(usage.loc[adata.obs_names, prog], errors="coerce").to_numpy()
        else:
            u = pd.to_numeric(usage[prog], errors="coerce").to_numpy()

        dfp["usage"] = u[pre_mask.to_numpy()]

        dfp["response_status"] = (
            dfp["response_status"]
            .astype("string")
            .str.strip()
            .str.lower()
        )
        dfp = dfp[dfp["response_status"].isin(group_order)].copy()

        dfp["sample_name"] = pd.Categorical(
            dfp["sample_name"],
            categories=sample_order_pre,
            ordered=True,
        )

        sample_mean = (
            dfp.groupby(["sample_name", "response_status"], observed=True)["usage"]
            .mean()
            .rename("mean_usage")
            .reset_index()
        )

        n_cells = (
            dfp.groupby("sample_name", observed=True)
            .size()
            .reindex(sample_order_pre)
            .fillna(0)
            .astype(int)
        )

        if dfp["usage"].dropna().empty:
            print(f"[skip] {prog}: no usable data after Pre + response_status filtering")
            continue

        fig_w = 1.25 * len(sample_order_pre) if len(sample_order_pre) > 0 else 8
        fig, ax = plt.subplots(figsize=(fig_w, 6.2))

        sns.violinplot(
            data=dfp,
            x="sample_name",
            y="usage",
            hue="response_status",
            palette=palette,
            order=sample_order_pre,
            hue_order=group_order,
            cut=0,
            inner=None,
            linewidth=0,
            dodge=True,
            ax=ax,
        )
        _set_violin_alpha(ax, alpha=0.35)

        sns.boxplot(
            data=dfp,
            x="sample_name",
            y="usage",
            hue="response_status",
            palette=palette,
            order=sample_order_pre,
            hue_order=group_order,
            showfliers=True,
            width=0.25,
            linewidth=1.1,
            flierprops=dict(
                marker="o",
                markersize=4,
                markerfacecolor="none",
                markeredgecolor="gray",
                alpha=0.6,
            ),
            ax=ax,
        )

        sns.scatterplot(
            data=sample_mean,
            x="sample_name",
            y="mean_usage",
            hue="response_status",
            palette=palette,
            hue_order=group_order,
            s=65,
            edgecolor="black",
            linewidth=0.9,
            legend=False,
            ax=ax,
            zorder=6,
        )

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles[:2],
            labels[:2],
            title="response_status",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=True,
        )

        ax.tick_params(axis="x", rotation=90)
        ax.set_xticks(range(len(sample_order_pre)))
        ax.set_xticklabels([f"{s}\n(n={n_cells.loc[s]})" for s in sample_order_pre])

        ax.set_title(f"{prog} (Pre)", fontsize=14, pad=12)
        ax.set_xlabel("")
        ax.set_ylabel("Per-cell program usage")
        fig.tight_layout()

        if save_pdf:
            out_path = out_dir / f"{safe_filename(prog)}__Pre_violin_box_meandot.pdf"
            fig.savefig(out_path, format="pdf", bbox_inches="tight")
            print(f"[saved] {out_path}")

        if show_plots:
            plt.show()

        plt.close(fig)


plot_pre_program_usage_panels(
    adata=adata,
    usage=usage,
    progs=progs,
    sample_order=sample_order,
    out_dir=PRE_PANEL_OUT_DIR,
    palette=palette,
    group_order=group_order,
    show_plots=SHOW_PLOTS,
    save_pdf=SAVE_PDF,
)
